In [ ]:

import json
import hashlib
import sys
from pathlib import Path

project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
_original_sys_path = list(sys.path)
sys.path.insert(0, str(project_root))
try:
    from energy_core import default_settings, make_batch, cpu_limits
    from energy_runner import run_scenario, run_benchmark
finally:
    sys.path[:] = _original_sys_path

_nb_path = project_root / "source" / "original.ipynb"
with open(_nb_path, "rb") as f:
    _nb_bytes = f.read()
_source_hash = hashlib.sha256(_nb_bytes).hexdigest()
assert _source_hash == "f7ea554af73cb21b3eb8c0569c7c21eac0b327e0ce35c29cf23806d74af926b8", f"unexpected source hash: {_source_hash}"

settings = default_settings()



In [ ]:

result = run_scenario(settings)

assert result["status"] == "optimal", f"expected optimal, got {result['status']}"
assert abs(result["objective"] - 21879.0) < 1e-4, f"objective mismatch: {result['objective']}"
assert result["modules"] == 30, f"modules mismatch: {result['modules']}"
_expected_dispatch = [4000.0, 6000.0, 5000.0, 800.0]
for i, (a, b) in enumerate(zip(result["dispatch"], _expected_dispatch)):
    assert abs(a - b) < 1e-4, f"dispatch[{i}] mismatch: {a} vs {b}"
_expected_active = [20, 30, 25, 4]
for i, (a, b) in enumerate(zip(result["active_modules"], _expected_active)):
    assert abs(a - b) < 1e-4, f"active[{i}] mismatch: {a} vs {b}"



In [ ]:

infeasible_settings = dict(settings)
infeasible_settings["max_modules"] = 20
infeasible_result = run_scenario(infeasible_settings)

assert infeasible_result["status"] == "infeasible", f"expected infeasible, got {infeasible_result['status']}"
assert infeasible_result["objective"] is None, f"expected None objective, got {infeasible_result['objective']}"
assert infeasible_result["dispatch"] == [], f"expected empty dispatch, got {infeasible_result['dispatch']}"

batch = make_batch(settings, 4)
n_workers = min(2, cpu_limits()["effective_cpus"])
benchmark_report = run_benchmark(batch, n_workers)

assert benchmark_report["comparison"]["matches"] is True, "comparison does not match"
assert benchmark_report["sequential"]["batch"] == batch, "sequential batch mismatch"
assert benchmark_report["parallel"]["batch"] == batch, "parallel batch mismatch"
assert len(benchmark_report["sequential"]["rows"]) == 4, f"expected 4 cases, got {len(benchmark_report['sequential']['rows'])}"
assert len(benchmark_report["parallel"]["rows"]) == 4, f"expected 4 cases, got {len(benchmark_report['parallel']['rows'])}"



In [ ]:

def _round6(v):
    if v is None:
        return None
    return round(float(v), 6)

_seq_rows = benchmark_report["sequential"]["rows"]
_batch_statuses = [r["result"]["status"] for r in _seq_rows]
_batch_objectives = [_round6(r["result"]["objective"]) for r in _seq_rows]

canonical = {
    "sourcehash": _source_hash,
    "default": {
        "status": result["status"],
        "objective": _round6(result["objective"]),
        "modules": int(result["modules"]),
        "dispatch": [_round6(x) for x in result["dispatch"]],
        "active": [_round6(x) for x in result["active_modules"]],
    },
    "infeasible": {
        "status": infeasible_result["status"],
    },
    "batch": {
        "case_count": 4,
        "statuses": _batch_statuses,
        "objectives": _batch_objectives,
    },
    "same_work_verified": True,
}

out_path = Path.cwd() / "results.json"
with open(out_path, "w") as f:
    json.dump({"results": canonical}, f, allow_nan=False, sort_keys=True)

print(json.dumps({"results": canonical}, allow_nan=False, sort_keys=True))